# LeetCode 1381 — Design a Stack With Increment Operation bằng Python

Link đề: https://leetcode.com/problems/design-a-stack-with-increment-operation/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Design

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class CustomStack:
    def __init__(self, maxSize: int):
        ...
    def push(self, x: int) -> None:
        ...
    def pop(self) -> int:
        ...
    def increment(self, k: int, val: int) -> None:
        ...
```

Trong notebook ta tự dựng **bộ chạy giả lập** chuỗi lệnh kiểu LeetCode (`["CustomStack","push",...]`, `[[3],[1],...]`) để test — đây là bản giả lập, không phải trình chấm thật.

## 1. Mục tiêu học

1. Thiết kế một lớp stack có giới hạn kích thước với ba thao tác `push`, `pop`, `increment(k, val)` (cộng `val` cho `k` phần tử **dưới đáy**).
2. Viết bản mô phỏng trực tiếp (`increment` tốn `O(k)`) và hiểu khi nào nó đủ tốt.
3. Nắm kỹ thuật **lazy propagation (cập nhật lười)** trên stack: ghi "món nợ" tại **một** vị trí, chỉ trả khi `pop`, đạt `O(1)` cho mọi thao tác.
4. Hiểu invariant của mảng `inc`: giá trị thật của phần tử `i` = `stack[i] + (tổng inc[j] với j ≥ i)`.
5. Tự viết bộ chạy lệnh kiểu LeetCode để kiểm thử các lớp thiết kế, đối chiếu ngẫu nhiên giữa các bản cài đặt.

## 2. Đề bài & ví dụ

Thiết kế stack hỗ trợ phép tăng giá trị:
- `CustomStack(maxSize)`: khởi tạo, `maxSize` là số phần tử tối đa.
- `push(x)`: thêm `x` lên đỉnh **nếu** stack chưa đầy (`< maxSize` phần tử); nếu đầy thì bỏ qua.
- `pop()`: lấy ra và trả về phần tử đỉnh, hoặc `-1` nếu stack rỗng.
- `increment(k, val)`: cộng `val` cho `k` phần tử **ở đáy**; nếu stack có ít hơn `k` phần tử thì cộng cho tất cả.

**Ví dụ:**
```
["CustomStack","push","push","pop","push","push","push","increment","increment","pop","pop","pop","pop"]
[[3],[1],[2],[],[2],[3],[4],[5,100],[2,100],[],[],[],[]]
→ [null,null,null,2,null,null,null,null,null,103,202,201,-1]
```
Diễn biến: `[1]` → `[1,2]` → pop 2 → `[1]` → `[1,2]` → `[1,2,3]` → push 4 bị bỏ (đầy) → `increment(5,100)`: `[101,102,103]` → `increment(2,100)`: `[201,202,103]` → pop 103, 202, 201, rồi `-1`.

**Ràng buộc:** `1 <= maxSize, x, k <= 1000`; `0 <= val <= 100`; mỗi phương thức `increment`, `push`, `pop` được gọi tối đa 1000 lần.

In [1]:
import random
import time

EX_OPS = ["CustomStack", "push", "push", "pop", "push", "push", "push", "increment", "increment", "pop", "pop", "pop", "pop"]
EX_ARGS = [[3], [1], [2], [], [2], [3], [4], [5, 100], [2, 100], [], [], [], []]
EX_OUT = [None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1]

def run(cls, ops, args):
    """Bộ chạy GIẢ LẬP kiểu LeetCode: lệnh đầu là hàm tạo, các lệnh sau gọi phương thức."""
    obj = cls(*args[0])
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out

print(len(EX_OPS), "lệnh; kết quả mong đợi:", EX_OUT)

13 lệnh; kết quả mong đợi: [None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1]


## 3. Kiến thức nền

### 3.1 List Python như một stack

`append(x)` thêm vào cuối (đỉnh), `pop()` lấy phần tử cuối — cả hai `O(1)` khấu hao. `len(st)` cho số phần tử, `st[-1]` là đỉnh, `st[0]` là đáy.

In [2]:
st = []
st.append(1); st.append(2); st.append(3)
print("stack:", st, "| đỉnh:", st[-1], "| đáy:", st[0])
print("pop ->", st.pop(), "| còn:", st)

stack: [1, 2, 3] | đỉnh: 3 | đáy: 1
pop -> 3 | còn: [1, 2]


### 3.2 Lớp (class), `__init__`, thuộc tính `self`

Bài thiết kế yêu cầu viết lớp với trạng thái lưu trong `self`. Mỗi đối tượng có trạng thái riêng.

In [3]:
class Counter:
    def __init__(self, start):
        self.value = start
    def add(self, d):
        self.value += d
        return self.value

c1, c2 = Counter(0), Counter(100)
c1.add(5)
print(c1.value, c2.value)

5 100


### 3.3 `getattr` — gọi phương thức theo tên chuỗi

LeetCode mô tả test bằng tên phương thức dạng chuỗi. `getattr(obj, "push")` trả về phương thức `obj.push`, gọi được như hàm. Đây là lõi của bộ chạy giả lập `run` ở trên.

In [4]:
c = Counter(10)
method = getattr(c, "add")
print(method(7), getattr(c, "value"))

17 17


### 3.4 `min` để "kẹp" chỉ số

`increment(k, val)` phải xử lý `k` lớn hơn kích thước stack ⇒ dùng `min(k, len(st))`. Thói quen "kẹp" (clamp) chỉ số giúp tránh `IndexError`.

In [5]:
st = [1, 2, 3]
for k in [1, 3, 5]:
    print(f"k={k} -> cộng cho {min(k, len(st))} phần tử đáy: chỉ số 0..{min(k, len(st)) - 1}")

k=1 -> cộng cho 1 phần tử đáy: chỉ số 0..0
k=3 -> cộng cho 3 phần tử đáy: chỉ số 0..2
k=5 -> cộng cho 3 phần tử đáy: chỉ số 0..2


### 3.5 Ý tưởng "món nợ" (lazy) và mảng hiệu

Thay vì cộng ngay cho nhiều phần tử, ta **ghi lại** một con số ở biên rồi "đẩy dần" khi cần đọc. Ví dụ quen thuộc là **mảng hiệu**: cộng `v` cho đoạn `[l, r]` bằng `d[l] += v; d[r+1] -= v`, giá trị thật là tổng tiền tố của `d`. Ở bài này chiều "đẩy" là **từ đỉnh xuống đáy** — đúng chiều mà `pop` đi qua.

In [6]:
d = [0] * 6
d[1] += 10; d[4] -= 10      # cộng 10 cho đoạn [1, 3]
vals, run_sum = [], 0
for x in d[:5]:
    run_sum += x; vals.append(run_sum)
print("mảng hiệu:", d, "-> giá trị thật:", vals)

mảng hiệu: [0, 10, 0, 0, -10, 0] -> giá trị thật: [0, 10, 10, 10, 0]


## 4. Ý tưởng nền tảng / trực giác

Theo dõi ví dụ của đề với "lời giải tay":

| Lệnh | Stack sau lệnh | Trả về |
|---|---|---|
| CustomStack(3) | [] | null |
| push(1), push(2) | [1, 2] | null |
| pop() | [1] | 2 |
| push(2), push(3) | [1, 2, 3] | null |
| push(4) | [1, 2, 3] (đầy, bỏ qua) | null |
| increment(5, 100) | [101, 102, 103] | null |
| increment(2, 100) | [201, 202, 103] | null |
| pop() ×3 | [] | 103, 202, 201 |
| pop() | [] | -1 |

Quan sát quan trọng cho tối ưu: **ta chỉ cần biết giá trị thật của một phần tử đúng lúc nó bị `pop`**. Cộng 100 cho cả `k` phần tử ngay lập tức là "làm trước" công việc mà có thể rất lâu sau mới cần.

## 5. Cách 1 — Brute Force / baseline: mô phỏng trực tiếp

**Ý tưởng:** list Python; `increment` lặp qua `min(k, len)` phần tử đáy và cộng `val`.

In [7]:
class CustomStackSim:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.st.append(x)

    def pop(self) -> int:
        return self.st.pop() if self.st else -1

    def increment(self, k: int, val: int) -> None:
        for i in range(min(k, len(self.st))):
            self.st[i] += val

got = run(CustomStackSim, EX_OPS, EX_ARGS)
print(got, "OK" if got == EX_OUT else "SAI")

[None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1] OK


**Độ phức tạp:** `push`, `pop`: `O(1)`; `increment`: `O(min(k, size))` ≤ `O(maxSize)`. Bộ nhớ `O(maxSize)`.

Với ràng buộc (≤ 1000 lần gọi mỗi loại, `k ≤ 1000`) tổng tối đa ~`10^6` phép — **đủ qua** LeetCode. Nhưng nếu đề nới ràng buộc (ví dụ `10^5` lệnh, `k ≤ 10^5`), `increment` thành nút thắt `O(k)`: `10^{10}` phép.

**Nút thắt:** cộng vào `k` ô trong khi chỉ cần biết giá trị khi `pop`.

In [8]:
def stress(cls, n_ops, cap, seed=0):
    rng = random.Random(seed)
    s = cls(cap)
    for _ in range(cap):
        s.push(1)
    t0 = time.perf_counter()
    for _ in range(n_ops):
        s.increment(cap, 1)
    return time.perf_counter() - t0

for cap in [1000, 4000, 16000]:
    print(f"cap={cap:5d}, 2000 lần increment(cap): {stress(CustomStackSim, 2000, cap):.3f}s")

cap= 1000, 2000 lần increment(cap): 0.076s


cap= 4000, 2000 lần increment(cap): 0.328s


cap=16000, 2000 lần increment(cap): 1.386s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `increment(k, val)` tác động lên **tiền tố đáy** `[0, m−1]` với `m = min(k, size)`. Khi `pop`, ta đi từ **đỉnh xuống**. Vậy ghi `val` vào **một ô** `inc[m−1]` (phần tử cao nhất bị ảnh hưởng) và quy ước:

> **Giá trị thật của phần tử `i` = `st[i] + Σ inc[j]` với mọi `j ≥ i` (j đang còn trong stack).**

Khi `pop` phần tử đỉnh `i`:
- Trả `st[i] + inc[i]` (vì mọi `j > i` không còn, phần "nợ" phía trên đã được dồn xuống `inc[i]` trước đó).
- **Dồn nợ xuống:** `inc[i−1] += inc[i]` (các phần tử dưới vẫn phải nhận khoản này).
- **Xóa nợ:** `inc[i] = 0` (để phần tử mới push vào vị trí `i` sau này không "thừa hưởng" nhầm).

**Quy tắc "nếu… thì…":**
- Nếu stack rỗng thì `increment` không làm gì, `pop` trả −1.
- Nếu `k ≥ size` thì ghi vào `inc[size−1]` (đỉnh).

Mọi thao tác thành `O(1)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — "Nhật ký increment" (bước trung gian để tìm ra lazy)

Ghi mỗi `increment` vào nhật ký `(thời điểm, k, val)`; mỗi phần tử lưu thời điểm được push. Khi `pop` phần tử ở chỉ số `idx`, cộng các `val` của những increment xảy ra **sau** khi nó được push và có `k > idx`. `increment` `O(1)` nhưng `pop` `O(số increment)` — chưa tốt, nhưng cho thấy: thông tin cần thiết chỉ là **tổng các val có k > idx** — gợi ý gom về một ô.

In [9]:
class CustomStackLog:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []          # (giá trị gốc, thời điểm push)
        self.log = []         # (thời điểm, k, val)
        self.clock = 0

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.clock += 1
            self.st.append((x, self.clock))

    def pop(self) -> int:
        if not self.st:
            return -1
        idx = len(self.st) - 1
        x, t0 = self.st.pop()
        return x + sum(v for t, k, v in self.log if t > t0 and k > idx)

    def increment(self, k: int, val: int) -> None:
        self.clock += 1
        self.log.append((self.clock, k, val))

got = run(CustomStackLog, EX_OPS, EX_ARGS)
print(got, "OK" if got == EX_OUT else "SAI")

[None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1] OK


Code nộp LeetCode (Cách 2 — chạy được với ràng buộc gốc vì ≤ 1000 increment):

In [10]:
class CustomStack:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []
        self.log = []
        self.clock = 0

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.clock += 1
            self.st.append((x, self.clock))

    def pop(self) -> int:
        if not self.st:
            return -1
        idx = len(self.st) - 1
        x, t0 = self.st.pop()
        return x + sum(v for t, k, v in self.log if t > t0 and k > idx)

    def increment(self, k: int, val: int) -> None:
        self.clock += 1
        self.log.append((self.clock, k, val))

print(run(CustomStack, EX_OPS, EX_ARGS) == EX_OUT)

True


**Độ phức tạp Cách 2:** `push`, `increment`: `O(1)`; `pop`: `O(L)` với `L` = số increment đã gọi; bộ nhớ `O(maxSize + L)`.

### 7.2 Cách 3 — Lazy increment: mảng `inc` cộng dồn khi pop (tối ưu)

**Từng đoạn:**
1. `st` là list giá trị gốc; `inc = [0] * maxSize`.
2. `increment(k, val)`: nếu stack khác rỗng, `inc[min(k, size) − 1] += val`.
3. `pop()`: `i = size − 1`; kết quả `st.pop() + inc[i]`; nếu `i > 0` thì `inc[i−1] += inc[i]`; `inc[i] = 0`.

In [11]:
class CustomStackLazy:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []
        self.inc = [0] * maxSize

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.st.append(x)

    def pop(self) -> int:
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]     # dồn nợ xuống
        self.inc[i] = 0                        # xóa nợ của ô này
        return res

    def increment(self, k: int, val: int) -> None:
        if self.st:
            self.inc[min(k, len(self.st)) - 1] += val

got = run(CustomStackLazy, EX_OPS, EX_ARGS)
print(got, "OK" if got == EX_OUT else "SAI")

[None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1] OK


Code nộp LeetCode (Cách 3):

In [12]:
class CustomStack:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []
        self.inc = [0] * maxSize

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.st.append(x)

    def pop(self) -> int:
        if not self.st:
            return -1
        i = len(self.st) - 1
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        res = self.st.pop() + self.inc[i]
        self.inc[i] = 0
        return res

    def increment(self, k: int, val: int) -> None:
        if self.st:
            self.inc[min(k, len(self.st)) - 1] += val

print(run(CustomStack, EX_OPS, EX_ARGS) == EX_OUT)

True


**Độ phức tạp Cách 3:** mọi thao tác `O(1)`; bộ nhớ `O(maxSize)`.

### 7.3 Cách 4 — Mảng cố định + con trỏ `top` (phong cách C/Java, không dùng append/pop)

Cùng ý tưởng lazy nhưng cấp phát sẵn hai mảng dài `maxSize` và giữ `top` = số phần tử. Hữu ích khi phỏng vấn yêu cầu "không dùng cấu trúc động".

In [13]:
class CustomStackArray:
    def __init__(self, maxSize: int):
        self.vals = [0] * maxSize
        self.inc = [0] * maxSize
        self.top = 0                         # số phần tử hiện có

    def push(self, x: int) -> None:
        if self.top < len(self.vals):
            self.vals[self.top] = x
            self.inc[self.top] = 0           # chắc chắn ô sạch
            self.top += 1

    def pop(self) -> int:
        if self.top == 0:
            return -1
        self.top -= 1
        i = self.top
        res = self.vals[i] + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        self.inc[i] = 0
        return res

    def increment(self, k: int, val: int) -> None:
        if self.top:
            self.inc[min(k, self.top) - 1] += val

got = run(CustomStackArray, EX_OPS, EX_ARGS)
print(got, "OK" if got == EX_OUT else "SAI")

[None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1] OK


Code nộp LeetCode (Cách 4):

In [14]:
class CustomStack:
    def __init__(self, maxSize: int):
        self.vals = [0] * maxSize
        self.inc = [0] * maxSize
        self.top = 0

    def push(self, x: int) -> None:
        if self.top < len(self.vals):
            self.vals[self.top] = x
            self.inc[self.top] = 0
            self.top += 1

    def pop(self) -> int:
        if self.top == 0:
            return -1
        self.top -= 1
        i = self.top
        res = self.vals[i] + self.inc[i]
        if i:
            self.inc[i - 1] += self.inc[i]
        self.inc[i] = 0
        return res

    def increment(self, k: int, val: int) -> None:
        if self.top:
            self.inc[min(k, self.top) - 1] += val

print(run(CustomStack, EX_OPS, EX_ARGS) == EX_OUT)

True


**Độ phức tạp Cách 4:** mọi thao tác `O(1)`, bộ nhớ `O(maxSize)` cấp phát một lần.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ của đề. Mỗi dòng in lệnh, kết quả, `st` (giá trị gốc), `inc`, và "giá trị thật" tính theo invariant `st[i] + Σ_{j≥i} inc[j]`:

| Lệnh | st | inc | giá trị thật | trả về |
|---|---|---|---|---|
| push 1, push 2 | [1,2] | [0,0,0] | [1,2] | – |
| pop | [1] | [0,0,0] | [1] | 2 |
| push 2, push 3, push 4 (bỏ) | [1,2,3] | [0,0,0] | [1,2,3] | – |
| increment(5,100) | [1,2,3] | [0,0,100] | [101,102,103] | – |
| increment(2,100) | [1,2,3] | [0,100,100] | [201,202,103] | – |
| pop | [1,2] | [0,200,0] | [201,202] | 103 |
| pop | [1] | [200,0,0] | [201] | 202 |
| pop | [] | [0,0,0] | [] | 201 |
| pop | [] | [0,0,0] | [] | -1 |

In [15]:
def real_values(s):
    out, acc = [], 0
    for i in range(len(s.st) - 1, -1, -1):
        acc += s.inc[i]
        out.append(s.st[i] + acc)
    return out[::-1]

def trace_custom_stack(ops, args):
    s = CustomStackLazy(*args[0])
    print(f"{'lệnh':18s} | {'trả về':6s} | {'st':12s} | {'inc':16s} | giá trị thật")
    for op, a in zip(ops[1:], args[1:]):
        r = getattr(s, op)(*a)
        name = f"{op}({', '.join(map(str, a))})"
        print(f"{name:18s} | {str(r):6s} | {str(s.st):12s} | {str(s.inc):16s} | {real_values(s)}")

trace_custom_stack(EX_OPS, EX_ARGS)

lệnh               | trả về | st           | inc              | giá trị thật
push(1)            | None   | [1]          | [0, 0, 0]        | [1]
push(2)            | None   | [1, 2]       | [0, 0, 0]        | [1, 2]
pop()              | 2      | [1]          | [0, 0, 0]        | [1]
push(2)            | None   | [1, 2]       | [0, 0, 0]        | [1, 2]
push(3)            | None   | [1, 2, 3]    | [0, 0, 0]        | [1, 2, 3]
push(4)            | None   | [1, 2, 3]    | [0, 0, 0]        | [1, 2, 3]
increment(5, 100)  | None   | [1, 2, 3]    | [0, 0, 100]      | [101, 102, 103]
increment(2, 100)  | None   | [1, 2, 3]    | [0, 100, 100]    | [201, 202, 103]
pop()              | 103    | [1, 2]       | [0, 200, 0]      | [201, 202]
pop()              | 202    | [1]          | [200, 0, 0]      | [201]
pop()              | 201    | []           | [0, 0, 0]        | []
pop()              | -1     | []           | [0, 0, 0]        | []


Quan sát: sau `increment(2,100)`, `inc = [0,100,100]`: ô 2 mang khoản 100 cho cả ba phần tử dưới nó (0..2), ô 1 mang khoản 100 cho phần tử 0..1. Khi pop đỉnh (chỉ số 2): trả `3 + 100 = 103`, dồn 100 xuống ô 1 ⇒ `inc[1] = 200`.

## 9. Code nộp LeetCode

In [16]:
class CustomStack:
    def __init__(self, maxSize: int):
        self.cap = maxSize
        self.st = []                    # giá trị gốc
        self.inc = [0] * maxSize        # inc[i]: khoản cộng cho mọi phần tử chỉ số <= i

    def push(self, x: int) -> None:
        if len(self.st) < self.cap:
            self.st.append(x)

    def pop(self) -> int:
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        self.inc[i] = 0
        return res

    def increment(self, k: int, val: int) -> None:
        if self.st:
            self.inc[min(k, len(self.st)) - 1] += val

got = run(CustomStack, EX_OPS, EX_ARGS)
print(got, "OK" if got == EX_OUT else "SAI")

[None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1] OK


## 10. Vì sao thuật toán đúng

**Invariant (I):** tại mọi thời điểm, với mọi chỉ số `i` đang có trong stack, giá trị thật của phần tử `i` = `st[i] + Σ_{j=i}^{size−1} inc[j]`; và `inc[j] = 0` với mọi `j ≥ size`.

- **Khởi tạo:** stack rỗng, `inc` toàn 0 ⇒ (I) đúng.
- **push(x):** phần tử mới ở chỉ số `size` có `inc[size] = 0` (theo vế sau của (I)) và không có `j > size` khác 0 ⇒ giá trị thật = `x`. Các phần tử cũ không đổi tổng.
- **increment(k, val):** với `m = min(k, size)`, `inc[m−1] += val` làm tổng `Σ_{j≥i}` tăng `val` **đúng** cho các `i ≤ m−1` (đáy `m` phần tử), không đổi cho `i ≥ m`.
- **pop():** đỉnh `i = size−1` có tổng `Σ_{j≥i} inc[j] = inc[i]` ⇒ trả về đúng. Sau `inc[i−1] += inc[i]; inc[i] = 0`, với mọi `i' < i` tổng `Σ_{j≥i'}` không đổi, và ô `i` (giờ ≥ size) về 0 ⇒ (I) vẫn đúng.

Vậy mọi giá trị `pop` trả ra đều là giá trị thật.

## 11. Phân tích Time / Space

Gọi `C = maxSize`, `L` = số lần gọi `increment`.

| Cách | push | pop | increment | Space |
|---|---|---|---|---|
| 1. Mô phỏng | `O(1)` | `O(1)` | `O(min(k, size))` | `O(C)` |
| 2. Nhật ký | `O(1)` | `O(L)` | `O(1)` | `O(C + L)` |
| 3. Lazy `inc` | `O(1)` | `O(1)` | `O(1)` | `O(C)` |
| 4. Mảng cố định | `O(1)` | `O(1)` | `O(1)` | `O(C)` |

Với ràng buộc gốc (≤ 1000 lệnh mỗi loại), cả bốn cách đều qua; Cách 3/4 là đáp án "chuẩn phỏng vấn" vì `O(1)` mọi thao tác.

## 12. Bộ test

### 12.1 Ví dụ đề + các kịch bản biên (qua bộ chạy giả lập)

In [17]:
CLASSES = {"sim": CustomStackSim, "log": CustomStackLog, "lazy": CustomStackLazy,
           "array": CustomStackArray, "submit": CustomStack}

scenarios = [
    (EX_OPS, EX_ARGS, EX_OUT),
    # pop trên stack rỗng, increment trên stack rỗng
    (["CustomStack", "pop", "increment", "pop", "push", "pop"], [[1], [], [3, 5], [], [7], []], [None, -1, None, -1, None, 7]),
    # maxSize = 1: push thứ hai bị bỏ
    (["CustomStack", "push", "push", "increment", "pop", "pop"], [[1], [5], [6], [1, 10], [], []], [None, None, None, None, 15, -1]),
    # nợ không được "thừa hưởng" bởi phần tử push sau khi pop
    (["CustomStack", "push", "push", "increment", "pop", "push", "pop", "pop"], [[3], [1], [2], [2, 10], [], [5], [], []],
     [None, None, None, None, 12, None, 5, 11]),
    # val = 0
    (["CustomStack", "push", "increment", "pop"], [[2], [4], [1, 0], []], [None, None, None, 4]),
]
for name, cls in CLASSES.items():
    for ops, args, want in scenarios:
        got = run(cls, ops, args)
        assert got == want, (name, ops, got, want)
print("Tất cả", len(CLASSES), "lớp đúng trên", len(scenarios), "kịch bản")

Tất cả 5 lớp đúng trên 5 kịch bản


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, đúng ràng buộc)

In [18]:
def random_script(rng, n_ops):
    cap = rng.randint(1, 8)
    ops, args = ["CustomStack"], [[cap]]
    for _ in range(n_ops):
        r = rng.random()
        if r < 0.4:
            ops.append("push"); args.append([rng.randint(1, 1000)])
        elif r < 0.7:
            ops.append("pop"); args.append([])
        else:
            ops.append("increment"); args.append([rng.randint(1, 10), rng.randint(0, 100)])
    return ops, args

rng = random.Random(1381)
for _ in range(800):
    ops, args = random_script(rng, rng.randint(1, 40))
    ref = run(CustomStackSim, ops, args)
    for name, cls in CLASSES.items():
        assert run(cls, ops, args) == ref, (name, ops, args)
print("800 kịch bản ngẫu nhiên: mọi lớp cho cùng kết quả với bản mô phỏng")

800 kịch bản ngẫu nhiên: mọi lớp cho cùng kết quả với bản mô phỏng


### 12.3 Test tải: ràng buộc gốc và ràng buộc "nới" (`C = 20000`)

In [19]:
for name in ["sim", "lazy", "array"]:
    print(f"{name:6s}: cap=1000 -> {stress(CLASSES[name], 1000, 1000):.4f}s | cap=20000, 1000 increment -> {stress(CLASSES[name], 1000, 20000):.4f}s")

sim   : cap=1000 -> 0.0369s | cap=20000, 1000 increment -> 0.7701s
lazy  : cap=1000 -> 0.0004s | cap=20000, 1000 increment -> 0.0001s
array : cap=1000 -> 0.0001s | cap=20000, 1000 increment -> 0.0001s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `increment` không kẹp `k` theo kích thước stack

In [20]:
class WrongNoClamp(CustomStackSim):
    def increment(self, k, val):
        for i in range(k):                 # SAI: k có thể > len(st)
            self.st[i] += val

try:
    print(run(WrongNoClamp, EX_OPS, EX_ARGS))
except IndexError as e:
    print("IndexError:", e, "<- tại increment(5, 100) khi stack chỉ có 3 phần tử")
print("đúng:", run(CustomStackSim, EX_OPS, EX_ARGS))

IndexError: list index out of range <- tại increment(5, 100) khi stack chỉ có 3 phần tử
đúng: [None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1]


**Vì sao sai:** đề nói rõ "nếu ít hơn `k` phần tử thì cộng cho tất cả". `range(k)` vượt quá chỉ số ⇒ `IndexError` (trên LeetCode là Runtime Error).

**Code đúng:** `for i in range(min(k, len(self.st)))` (hoặc `inc[min(k, size) − 1]`).

### Lỗi 2 — Lazy nhưng quên **dồn nợ xuống** khi pop

In [21]:
class WrongNoPropagate(CustomStackLazy):
    def pop(self):
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        self.inc[i] = 0                    # SAI: thiếu inc[i-1] += inc[i]
        return res

print("sai:", run(WrongNoPropagate, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai: [None, None, None, 2, None, None, None, None, None, 103, 102, 1, -1]
đúng: [None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1]


**Vì sao sai:** `inc[i]` là khoản cộng cho **mọi** phần tử chỉ số `≤ i`, không riêng phần tử `i`. Xóa mà không dồn xuống ⇒ các phần tử dưới mất khoản đó: `202` thành `102`, `201` thành `1`... (với `increment(5,100)` chỉ ghi ở ô đỉnh).

**Code đúng:** `if i > 0: inc[i−1] += inc[i]` trước khi xóa.

### Lỗi 3 — Dồn nợ nhưng quên **xóa** `inc[i]`

In [22]:
class WrongNoReset(CustomStackLazy):
    def pop(self):
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        # SAI: thiếu self.inc[i] = 0
        return res

ops = ["CustomStack", "push", "push", "increment", "pop", "push", "pop", "pop"]
args = [[3], [1], [2], [2, 10], [], [5], [], []]
print("sai:", run(WrongNoReset, ops, args))
print("đúng:", run(CustomStackSim, ops, args))

sai: [None, None, None, None, 12, None, 15, 21]
đúng: [None, None, None, None, 12, None, 5, 11]


**Vì sao sai:** sau khi pop phần tử ở chỉ số 1, ô `inc[1]` vẫn còn 10. Phần tử `5` push vào chỉ số 1 **thừa hưởng** khoản 10 không thuộc về nó ⇒ trả 15 thay vì 5 (và phần tử dưới bị cộng 2 lần).

**Code đúng:** `self.inc[i] = 0` sau khi dồn (Cách 4 còn xóa thêm khi push cho chắc).

### Lỗi 4 — `push` bỏ qua `maxSize`

In [23]:
class WrongNoCap(CustomStackLazy):
    def __init__(self, maxSize):
        super().__init__(maxSize)
        self.inc = [0] * 2000              # tránh IndexError để thấy kết quả sai
    def push(self, x):
        self.st.append(x)                  # SAI: không kiểm tra đầy

print("sai:", run(WrongNoCap, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai: [None, None, None, 2, None, None, None, None, None, 104, 103, 202, 201]
đúng: [None, None, None, 2, None, None, None, None, None, 103, 202, 201, -1]


**Vì sao sai:** `push(4)` lẽ ra bị bỏ vì stack đã có 3 phần tử. Push nhầm làm mọi kết quả `pop` sau đó lệch (pop đầu tiên trả `4 + ...` thay vì `103`).

**Code đúng:** `if len(self.st) < self.cap: self.st.append(x)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **design**: viết trước bản mô phỏng đơn giản để làm **chuẩn đối chiếu**, rồi mới tối ưu; luôn có bộ chạy lệnh kiểu LeetCode.
- "Cập nhật cả một **tiền tố đáy**, chỉ đọc ở **đỉnh**" ⇒ lazy: ghi ở biên trên của đoạn, dồn xuống khi pop.
- Lazy luôn có **ba** việc khi lấy phần tử ra: *cộng nợ vào kết quả* → *dồn nợ cho phần còn lại* → *xóa nợ ô cũ*.
- Kẹp chỉ số bằng `min(k, size)`; nhớ xử lý stack rỗng ở cả `pop` lẫn `increment`.
- Mẫu tương tự: **Min Stack** (lưu kèm min ở mỗi tầng), **Max Frequency Stack**, mảng hiệu cho cập nhật đoạn.

## 15. Biến thể của bài

1. **Increment `k` phần tử ở đỉnh** (thay vì đáy): đoạn `[size−k, size−1]` — có thể lazy bằng hai mốc nhưng cần cẩn thận với phần tử push sau.
2. **Increment đoạn bất kỳ `[l, r]` tính từ đáy:** `inc[r] += v; inc[l−1] −= v` (bài tập 5).
3. **Thêm `getMax()` / `getMin()`** trong `O(1)`: kết hợp với Min Stack — cần cẩn thận vì increment làm thay đổi min.
4. **Queue với increment tiền tố** (đầu hàng đợi): lazy theo chiều ngược.
5. **Segment tree với lazy propagation**: tổng quát hóa cho cập nhật/truy vấn đoạn tùy ý trên mảng.

## 16. So sánh các cách

| Cách | Ý tưởng | push / pop / increment | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | cộng trực tiếp `min(k,size)` ô | `O(1)` / `O(1)` / `O(k)` | `O(C)` | ràng buộc nhỏ, làm chuẩn đối chiếu |
| 2. Nhật ký | lưu increment, tính lúc pop | `O(1)` / `O(L)` / `O(1)` | `O(C+L)` | bước trung gian để hiểu lazy |
| 3. Lazy `inc` | ghi ở `inc[m−1]`, dồn khi pop | `O(1)` mọi thao tác | `O(C)` | **đáp án chuẩn** |
| 4. Mảng cố định | như 3 + con trỏ `top` | `O(1)` mọi thao tác | `O(C)` | khi cấm cấu trúc động / viết kiểu C |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** kết quả của chuỗi lệnh
`["CustomStack","push","push","push","increment","pop","increment","pop","pop","pop"]`, `[[2],[10],[20],[30],[1,5],[],[3,1],[],[],[]]`?

**Bài 2 (tính tay):** với Cách 3, viết mảng `inc` sau mỗi lệnh của Bài 1.

**Bài 3 (điền TODO):** hoàn thiện `pop` cho lớp lazy dưới đây.

In [24]:
class LazyTodo:
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    def __init__(self, maxSize):
        self.cap, self.st, self.inc = maxSize, [], [0] * maxSize
    def push(self, x):
        if len(self.st) < self.cap:
            self.st.append(x)
    def increment(self, k, val):
        if self.st:
            self.inc[min(k, len(self.st)) - 1] += val
    def pop(self):
        # TODO: trả -1 nếu rỗng; lấy i = đỉnh; res = st.pop() + inc[i];
        #       dồn inc[i] xuống inc[i-1] (nếu i > 0); xóa inc[i]; trả res
        pass

**Bài 4 (tự viết lại):** viết lại lớp lazy **không dùng list `st` động** mà dùng một list duy nhất chứa cặp `[giá trị, nợ]` cấp phát sẵn `maxSize` phần tử.

**Bài 5 (mini challenge):** viết `RangeStack` có thêm `incrementRange(l, r, val)`: cộng `val` cho các phần tử có chỉ số từ đáy `l..r` (0-based, chỉ những chỉ số đang tồn tại). Mọi thao tác `O(1)`.

In [25]:
class RangeStack:
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    def __init__(self, maxSize):
        pass  # TODO
    def push(self, x):
        pass  # TODO
    def pop(self):
        pass  # TODO
    def incrementRange(self, l, r, val):
        pass  # TODO

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | maxSize = 2: push 30 bị bỏ | sau `increment(1,5)`: `[15,20]` | `[null,null,null,null,null,20,null,16,-1,-1]` |
| 2 | `increment(1,5)` ghi vào `inc[0]` | pop đỉnh 1 (inc 0) | `[5,0]`, `[5,0]`, `[6,0]` (increment(3,1) ghi ô 0), `[0,0]` |
| 3 | Ba việc: cộng, dồn, xóa | dồn chỉ khi `i > 0` | xem Cách 3 |
| 4 | `data = [[0, 0] for _ in range(C)]`, `top` | push ghi `data[top] = [x, 0]` | pop: `v, d = data[top-1]`; dồn `d` xuống `data[top-2][1]` |
| 5 | Mảng hiệu theo chiều đỉnh → đáy | `inc[r'] += v`, `inc[l−1] −= v` với `r' = min(r, size−1)` | bỏ qua nếu `l > r'`; pop giống Cách 3 |

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ops1 = ["CustomStack", "push", "push", "push", "increment", "pop", "increment", "pop", "pop", "pop"]
args1 = [[2], [10], [20], [30], [1, 5], [], [3, 1], [], [], []]
ans1 = run(CustomStackLazy, ops1, args1)
print("Bài 1:", ans1)
assert ans1 == [None, None, None, None, None, 20, None, 16, -1, -1]

# Bài 2: in inc sau mỗi lệnh
s = CustomStackLazy(2)
incs = []
for op, a in zip(ops1[1:], args1[1:]):
    getattr(s, op)(*a)
    incs.append(list(s.inc))
print("Bài 2:", incs)
assert incs[3] == [5, 0] and incs[4] == [5, 0] and incs[5] == [6, 0] and incs[6] == [0, 0]

Bài 1: [None, None, None, None, None, 20, None, 16, -1, -1]
Bài 2: [[0, 0], [0, 0], [0, 0], [5, 0], [5, 0], [6, 0], [0, 0], [0, 0], [0, 0]]


In [27]:
# Bài 3
class LazySol(LazyTodo):
    def pop(self):
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        self.inc[i] = 0
        return res

rng = random.Random(3)
for _ in range(300):
    ops, args = random_script(rng, rng.randint(1, 30))
    assert run(LazySol, ops, args) == run(CustomStackSim, ops, args)
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4
class PairStack:
    def __init__(self, maxSize):
        self.data = [[0, 0] for _ in range(maxSize)]
        self.top = 0
    def push(self, x):
        if self.top < len(self.data):
            self.data[self.top] = [x, 0]
            self.top += 1
    def pop(self):
        if self.top == 0:
            return -1
        self.top -= 1
        v, d = self.data[self.top]
        if self.top > 0:
            self.data[self.top - 1][1] += d
        return v + d
    def increment(self, k, val):
        if self.top:
            self.data[min(k, self.top) - 1][1] += val

rng = random.Random(4)
for _ in range(300):
    ops, args = random_script(rng, rng.randint(1, 30))
    assert run(PairStack, ops, args) == run(CustomStackSim, ops, args)
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
class RangeStackSol:
    def __init__(self, maxSize):
        self.cap, self.st, self.inc = maxSize, [], [0] * maxSize
    def push(self, x):
        if len(self.st) < self.cap:
            self.st.append(x)
    def pop(self):
        if not self.st:
            return -1
        i = len(self.st) - 1
        res = self.st.pop() + self.inc[i]
        if i > 0:
            self.inc[i - 1] += self.inc[i]
        self.inc[i] = 0
        return res
    def incrementRange(self, l, r, val):
        r = min(r, len(self.st) - 1)
        if l > r:
            return
        self.inc[r] += val
        if l > 0:
            self.inc[l - 1] -= val

class RangeStackSim(CustomStackSim):
    def incrementRange(self, l, r, val):
        for i in range(l, min(r, len(self.st) - 1) + 1):
            self.st[i] += val

rng = random.Random(5)
for _ in range(500):
    cap = rng.randint(1, 8)
    ops, args = ["X"], [[cap]]
    for _ in range(rng.randint(1, 40)):
        r = rng.random()
        if r < 0.4: ops.append("push"); args.append([rng.randint(1, 1000)])
        elif r < 0.7: ops.append("pop"); args.append([])
        else:
            l = rng.randint(0, 8); ops.append("incrementRange"); args.append([l, rng.randint(l, 9), rng.randint(0, 100)])
    assert run(RangeStackSol, ops, args) == run(RangeStackSim, ops, args), (ops, args)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template lazy stack (cập nhật tiền tố đáy):**

```python
self.st, self.inc = [], [0] * maxSize
def increment(self, k, val):
    if self.st: self.inc[min(k, len(self.st)) - 1] += val   # ghi ở biên trên
def pop(self):
    if not self.st: return -1
    i = len(self.st) - 1
    res = self.st.pop() + self.inc[i]          # 1) cộng nợ
    if i: self.inc[i - 1] += self.inc[i]       # 2) dồn nợ xuống
    self.inc[i] = 0                            # 3) xóa nợ
    return res
```

**Checklist trước khi submit:**
- [ ] `push` kiểm tra `len < maxSize`.
- [ ] `pop` rỗng trả `-1`; `increment` rỗng không làm gì.
- [ ] `increment` kẹp `min(k, size)`.
- [ ] Lazy: đủ ba bước cộng – dồn – xóa khi pop.
- [ ] Đã đối chiếu ngẫu nhiên với bản mô phỏng qua bộ chạy lệnh.
- [ ] Đúng tên lớp/phương thức: `CustomStack`, `push`, `pop`, `increment(k, val)`.